In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys
import os
from datasets import load_dataset

root_path = os.path.abspath(os.path.join(".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from utils import standardize_model_names # noqa: E402

In [9]:
csv_url = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/ic_before_after.csv"
df_csv = pd.read_csv(csv_url)
df_csv = standardize_model_names(df_csv, model_column="model")

df_csv = df_csv[~df_csv["model"].str.contains("teacher", case=False, na=False)].copy()

In [11]:
dataset_meta = load_dataset(
    "mehmetdavut/RubyCraft-3.4-Eval-Logs", 
    "intrinsic_capability_humaneval", 
    split="train"
)
df_meta = dataset_meta.to_pandas()
df_meta = standardize_model_names(df_meta, model_column="model_architecture")
df_meta = df_meta[~df_meta["model_role"].str.contains("Teacher", case=False, na=False)].copy()

df_meta["passed_bool"] = df_meta["raw_avg_score"] > 0
meta_grouped = df_meta.groupby([
    "experiment_id", "Architecture", "dataset_size", "data_split", "quantization"
])["passed_bool"].sum().reset_index()
meta_grouped.rename(columns={"passed_bool": "raw_score"}, inplace=True)

In [12]:
meta_grouped = meta_grouped.sort_values(["experiment_id", "Architecture", "raw_score", "data_split"])
meta_grouped["rank"] = meta_grouped.groupby(["experiment_id", "Architecture"])["raw_score"].rank(method="first")

df_csv = df_csv.sort_values(["experiment", "Architecture", "before_passed", "model"])
df_csv["rank"] = df_csv.groupby(["experiment", "Architecture"])["before_passed"].rank(method="first")

df_merged = df_csv.merge(
    meta_grouped,
    left_on=["experiment", "Architecture", "rank"],
    right_on=["experiment_id", "Architecture", "rank"],
    how="left"
)

In [13]:
base_mask = df_merged["data_split"].isna() | (df_merged["data_split"] == "Base")
df_merged.loc[base_mask, "dataset_size"] = "-"
df_merged.loc[base_mask, "data_split"] = "Vanilla"
df_merged.loc[base_mask, "quantization"] = "-"

In [14]:
df_merged["after_passed"] = df_merged["after_passed"].fillna(0).astype(int)
df_merged["before_passed"] = df_merged["before_passed"].fillna(0).astype(int)

tier_col = "Tier_x" if "Tier_x" in df_merged.columns else "Tier"

df_solved = df_merged[[
    "experiment", "Architecture", tier_col, 
    "dataset_size", "data_split", "quantization", 
    "before_passed", "after_passed"
]].copy()

df_solved.rename(columns={tier_col: "Tier"}, inplace=True)
df_solved = df_solved.sort_values(by="after_passed", ascending=False).reset_index(drop=True)

display(df_solved.head(50))

,experiment,Architecture,Tier,dataset_size,data_split,quantization,before_passed,after_passed
0,exp-111,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
1,exp-113,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
2,exp-112,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
3,exp-114,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
4,exp-110,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
5,exp-109,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
6,exp-112,Qwen,Big (≥ 7B),-,Vanilla,-,52,86
7,exp-114,Qwen,Big (≥ 7B),-,Vanilla,-,52,86
8,exp-109,Qwen,Big (≥ 7B),-,Vanilla,-,52,86
9,exp-113,Qwen-Coder,Big (≥ 7B),1K,ALL,16-bit,73,86


In [15]:
df_big = df_solved[df_solved["Tier"] == "Big (≥ 7B)"].copy()
df_big = df_big.sort_values(by="after_passed", ascending=False).reset_index(drop=True)

display(df_big.head(30))

,experiment,Architecture,Tier,dataset_size,data_split,quantization,before_passed,after_passed
0,exp-111,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
1,exp-110,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
2,exp-109,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
3,exp-113,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
4,exp-114,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
5,exp-112,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,15,101
6,exp-112,Qwen,Big (≥ 7B),-,Vanilla,-,52,86
7,exp-114,Qwen,Big (≥ 7B),-,Vanilla,-,52,86
8,exp-109,Qwen,Big (≥ 7B),-,Vanilla,-,52,86
9,exp-113,Qwen-Coder,Big (≥ 7B),1K,ALL,16-bit,73,86


In [16]:
df_small = df_solved[df_solved["Tier"] == "Small (< 4B)"].copy()
df_small = df_small.sort_values(by="after_passed", ascending=False).reset_index(drop=True)

display(df_small.head(50))

,experiment,Architecture,Tier,dataset_size,data_split,quantization,before_passed,after_passed
0,exp-108,Qwen-Coder,Small (< 4B),5K,ALL,16-bit,67,73
1,exp-106,Qwen-Coder,Small (< 4B),5K,ALL,8-bit,61,66
2,exp-104,Qwen-Coder,Small (< 4B),5K,ALL,4-bit,56,64
3,exp-108,Qwen-Coder,Small (< 4B),5K,HQ,16-bit,54,63
4,exp-105,Qwen-Coder,Small (< 4B),1K,ALL,8-bit,62,62
5,exp-107,Qwen-Coder,Small (< 4B),1K,ALL,16-bit,62,62
6,exp-106,Qwen-Coder,Small (< 4B),5K,HQ,8-bit,49,60
7,exp-104,Qwen-Coder,Small (< 4B),5K,HQ,4-bit,51,58
8,exp-103,Qwen-Coder,Small (< 4B),1K,ALL,4-bit,54,58
9,exp-116,Qwen-Coder,Small (< 4B),1K,HQ,16-bit,45,56
